# Comparing LMC centering methods

This notebook compares the centering methods in `nba.com.CenterHalo` (`mean_pos`, `shrinking_sphere`, `shrinking_sphere_numba` and `min_potential`) on the LMC dark matter halo of the GC21 MWLMC5 simulation.

The orbits are computed beforehand with [compute_lmc_orbits.py](compute_lmc_orbits.py):

```
python compute_lmc_orbits.py /path/to/snapshots --outdir lmc_orbits
```

Loading one snapshot needs ~6-7 GB of memory, since `ReadGC21.read_halo` reads all dark matter particles before selecting the LMC.

In [ ]:
%matplotlib inline
import os

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from nba.ios import ReadGC21

## Configuration

In [ ]:
path = "/mnt/home/nico/ceph/gadget_runs/MWLMC/MWLMC5/out/"
snapname = "MWLMC5_100M_b0_vir_OM3_G4_{:03d}.hdf5"
orbit_dir = "lmc_orbits"

methods = ["mean_pos", "shrinking_sphere", "shrinking_sphere_numba", "min_potential"]

# Snapshots to plot. None: first, middle and last snapshot of the orbit files
snapshots_to_show = None

box = 50  # half size of the projected density maps [kpc]
nbins = 200

## Orbits

Each file has the columns `snap, time, x, y, z, vx, vy, vz`.

In [ ]:
orbits = {m: np.loadtxt(os.path.join(orbit_dir, f"lmc_orbit_{m}.txt"), ndmin=2) for m in methods}

snaps = orbits[methods[0]][:, 0].astype(int)
times = orbits[methods[0]][:, 1]
for m in methods:
    assert np.array_equal(orbits[m][:, 0].astype(int), snaps), f"{m} has different snapshots"

if snapshots_to_show is None:
    snapshots_to_show = sorted(set([snaps[0], snaps[len(snaps) // 2], snaps[-1]]))
print(f"{len(snaps)} snapshots ({snaps[0]}-{snaps[-1]}), showing {snapshots_to_show}")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4), sharex=True)
labels = ["x", "y", "z"]
for m in methods:
    pos = orbits[m][:, 2:5]
    for i in range(3):
        ax[i].plot(times, pos[:, i], label=m)
    ax[3].plot(times, np.linalg.norm(pos, axis=1), label=m)

for i, label in enumerate(labels + ["r"]):
    ax[i].set_xlabel("time")
    ax[i].set_ylabel(f"{label} [kpc]")
    for k in snapshots_to_show:
        ax[i].axvline(times[snaps == k][0], c="0.7", ls=":")
ax[3].legend()
plt.tight_layout()
plt.show()

Distance between the centers found by each method and the `shrinking_sphere_numba` center:

In [ ]:
reference = "shrinking_sphere_numba"
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
for m in methods:
    if m == reference:
        continue
    offset = np.linalg.norm(orbits[m][:, 2:5] - orbits[reference][:, 2:5], axis=1)
    ax.semilogy(times, offset, label=m)
ax.set_xlabel("time")
ax.set_ylabel(f"distance to {reference} center [kpc]")
ax.legend()
plt.show()

## Projected density of the LMC centered with each method

For each selected snapshot the LMC dark matter particles are loaded once and shifted to the center given by each method. Rows are the projections (x-y, x-z, y-z) and columns the methods. All panels of a snapshot share the same color scale; the cross marks the center.

In [ ]:
projections = [(0, 1), (0, 2), (1, 2)]
axis_names = ["x", "y", "z"]
edges = np.linspace(-box, box, nbins + 1)

for k in snapshots_to_show:
    lmc_pos = ReadGC21(path, snapname.format(k)).read_halo(['pos'], halo='LMC', ptype='dm')['pos']
    row = np.where(snaps == k)[0][0]

    # Projected histograms for every method and projection
    hists = {}
    for m in methods:
        centered = lmc_pos - orbits[m][row, 2:5]
        for i, j in projections:
            hists[m, i, j], _, _ = np.histogram2d(centered[:, i], centered[:, j], bins=[edges, edges])
    del lmc_pos

    norm = LogNorm(vmin=1, vmax=max(h.max() for h in hists.values()))

    fig, ax = plt.subplots(3, len(methods), figsize=(4 * len(methods), 12), sharex=True, sharey=True)
    for c, m in enumerate(methods):
        for r, (i, j) in enumerate(projections):
            ax[r][c].imshow(hists[m, i, j].T, origin='lower', extent=[-box, box, -box, box],
                            norm=norm, cmap='magma')
            ax[r][c].plot(0, 0, 'c+', ms=12, mew=1.5)
            ax[r][c].set_xlabel(f"{axis_names[i]} [kpc]")
            ax[r][c].set_ylabel(f"{axis_names[j]} [kpc]")
        ax[0][c].set_title(m)
    fig.suptitle(f"Snapshot {k}, time = {times[row]:.3f}", y=1.0)
    plt.tight_layout()
    plt.show()